# Single Experiment Analysis

This notebook demonstrates the end-to-end workflow for analysing a **single** neutron reflectometry experiment:

1. Load experimental Q/R/dR data
2. Apply error-bar preprocessing
3. Construct constraint-based prior bounds from known true parameters
4. Run posterior inference with the normalizing flow (NF) model
5. Compute Constraint MAPE metrics
6. Visualise the reflectivity fit and SLD profile

All steps use the existing pipeline modules — no logic is reimplemented here.

## 1. Setup

Add the project root to `sys.path` so the pipeline modules are importable from within the `notebooks/` subdirectory.

In [2]:
%matplotlib inline

import sys
import os

# Add project root to path when running from notebooks/ subdirectory
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"Device          : {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"Project root    : {PROJECT_ROOT}")


PyTorch version : 2.7.1+cu118
CUDA available  : True
Device          : cuda
Project root    : /home/levytskyi/Documents/TEMP_thesis_code_testing/evaluation_pipeline


## 2. Choose an Experiment

Set `EXPERIMENT_ID` to a sample identifier in `sXXXXXX` format (e.g. `"s000004"`).
Available IDs can be found by listing `dataset/test/`.
`LAYER_COUNT` describes the film layer structure used for prior construction and parameter parsing.

In [3]:
# --- user configuration ---
EXPERIMENT_ID = "s000004"   # sample identifier in sXXXXXX format
LAYER_COUNT   = 1           # 0, 1, or 2 layers

NF_CONFIG     = "example_nf_config_reflectorch.yaml"
NF_SAMPLES    = 1000
PRIORS_DEV    = 30      # constraint half-width as % of true value

# Preprocessing parameters
ENABLE_PREPROCESSING = True
ERROR_THRESHOLD      = 0.75
CONSECUTIVE          = 5

## 3. Discover Experiment Files

`parameter_discovery.discover_experiment_files` locates the reflectivity data file and the ground-truth model file for the chosen experiment.

In [4]:
from pathlib import Path
from config import DATA_DIRECTORY
from parameter_discovery import parse_true_parameters_from_model_file

data_dir   = Path(PROJECT_ROOT) / DATA_DIRECTORY
data_file  = data_dir / f"{EXPERIMENT_ID}_experimental_curve.dat"
model_file = data_dir / f"{EXPERIMENT_ID}_model.txt"

print(f"Data file  : {data_file}")
print(f"Model file : {model_file}")

# Parse true structural parameters from the model file
true_params_dict = parse_true_parameters_from_model_file(model_file)

layer_key   = f"{LAYER_COUNT}_layer"
param_names = true_params_dict[layer_key]["param_names"]
true_params  = true_params_dict[layer_key]["params"]

print(f"\nTrue parameters ({LAYER_COUNT}-layer):")
for name, val in zip(param_names, true_params):
    print(f"  {name:<20} = {val:.4f}")


Data file  : /home/levytskyi/Documents/TEMP_thesis_code_testing/evaluation_pipeline/dataset/test/s000004_experimental_curve.dat
Model file : /home/levytskyi/Documents/TEMP_thesis_code_testing/evaluation_pipeline/dataset/test/s000004_model.txt
Parsing true parameters from: /home/levytskyi/Documents/TEMP_thesis_code_testing/evaluation_pipeline/dataset/test/s000004_model.txt
Parsed layer 'fronting': sld=3.5e-06, thickness=inf, roughness=8.04
Parsed layer 'layer1': sld=3.56709e-06, thickness=28.63, roughness=7.04
Parsed layer 'backing': sld=9.79867e-06, thickness=inf, roughness=0.0
Parsed layers: ['fronting', 'layer1', 'backing']
Found 1 material layers: ['layer1']
Parsing as 1-layer model
1-layer parameters: [28.63, 8.04, 7.04, 3.56709, 9.79867]
SLD values converted to 10^-6 units - layer_sld: 3.57, sub_sld: 9.80

True parameters (1-layer):
  thickness            = 28.6300
  amb_rough            = 8.0400
  sub_rough            = 7.0400
  layer_sld            = 3.5671
  sub_sld            

## 4. Load and Preprocess Experimental Data

The preprocessing step truncates the Q-range at the first run of consecutive high relative-error points.
This prevents tensor concatenation errors during NF inference.

In [5]:
from simple_pipeline import load_experimental_data

q_exp, curve_exp, sigmas_exp = load_experimental_data(
    data_file_path=data_file,
    enable_preprocessing=ENABLE_PREPROCESSING,
    threshold=ERROR_THRESHOLD,
    consecutive=CONSECUTIVE,
)

# Quick visual inspection
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].semilogy(q_exp, curve_exp, "k.", ms=3, label="R(Q)")
axes[0].set_xlabel(r"$Q$ ($\AA^{-1}$)")
axes[0].set_ylabel(r"$R$")
axes[0].set_title("Reflectivity curve")
axes[0].legend()

rel_err = sigmas_exp / curve_exp
axes[1].plot(q_exp, rel_err, "r.", ms=3)
axes[1].axhline(ERROR_THRESHOLD, ls="--", color="grey", label=f"threshold = {ERROR_THRESHOLD}")
axes[1].set_xlabel(r"$Q$ ($\AA^{-1}$)")
axes[1].set_ylabel(r"$dR / R$")
axes[1].set_title("Relative error after preprocessing")
axes[1].legend()

plt.tight_layout()
plt.show()

Loading experimental data from: /home/levytskyi/Documents/TEMP_thesis_code_testing/evaluation_pipeline/dataset/test/s000004_experimental_curve.dat
Data shape: (76, 4)
Detected experimental data (4 columns) - using actual errors
Raw Q range: 0.0070 - 0.2835 Å⁻¹
Raw curve shape: (76,)
Raw relative error range: 0.0055 - 24.5210
Starting comprehensive data preprocessing
Initial data points: 76
Filtering high error points (threshold: 75.0%)
Found 10 points with high relative error
Truncating data at index 67 due to consecutive high-error points
Preprocessing complete:
  Original points: 76
  Final points: 67
  Removed: 9 (11.8%)
  Q range: 0.0070 - 0.1737 Å⁻¹
Final Q range: 0.0070 - 0.1737 Å⁻¹
Final curve shape: (67,)
Final relative error range: 0.0055 - 1.5706


/tmp/ipykernel_1326710/56885285.py:27: UserWarning: The figure layout has changed to tight
  plt.tight_layout()
/tmp/ipykernel_1326710/56885285.py:28: UserWarning: FigureCanvasPdf is non-interactive, and thus cannot be shown
  plt.show()


## 5. Build Constraint-Based Prior Bounds

The prior bounds are constructed by centring a window of ±`PRIORS_DEV`% of the true value around each true parameter.
The window is additionally clipped to the physical constraint ranges defined in `model_constraints.json`.

In [6]:
from parameter_discovery import get_prior_bounds_for_experiment

prior_bounds = get_prior_bounds_for_experiment(
    experiment_id=EXPERIMENT_ID,
    true_params_dict=true_params_dict,
    layer_count=LAYER_COUNT,
    priors_type="constraint_based",
    deviation=PRIORS_DEV / 100.0,
)

print("Prior bounds (constraint-based, deviation = {:.0%}):".format(PRIORS_DEV / 100.0))
print(f"{'Parameter':<20}  {'Min':>10}  {'True':>10}  {'Max':>10}")
print("-" * 56)
for name, val, (lo, hi) in zip(param_names, true_params, prior_bounds):
    print(f"{name:<20}  {lo:>10.4f}  {val:>10.4f}  {hi:>10.4f}")


Generating constraint_based prior bounds for s000004 (1 layer)
Generating constraint-based prior bounds (30% of constraint span)
  thickness: true=28.630, constraint_span=999.0, target_width=299.700 -> [1.000, 300.700]
  amb_rough: true=8.040, constraint_span=60.0, target_width=18.000 -> [0.000, 18.000]
  sub_rough: true=7.040, constraint_span=60.0, target_width=18.000 -> [0.000, 18.000]
  layer_sld: true=3.567, constraint_span=24.0, target_width=5.000 -> [1.067, 6.067]
  sub_sld: true=9.799, constraint_span=24.0, target_width=5.000 -> [7.299, 12.299]
Prior bounds (constraint-based, deviation = 30%):
Parameter                    Min        True         Max
--------------------------------------------------------
thickness                 1.0000     28.6300    300.7000
amb_rough                 0.0000      8.0400     18.0000
sub_rough                 0.0000      7.0400     18.0000
layer_sld                 1.0671      3.5671      6.0671
sub_sld                   7.2987      9.7987     1

## 6. Load Model and Run Inference

`EasyInferenceModel` loads the trained NF weights and config. `preprocess_and_sample` interpolates the data onto the model Q-grid and draws `NF_SAMPLES` posterior samples via importance-weighted sampling.

In [ ]:
from reflectorch import EasyInferenceModel
from simple_pipeline import _extend_prior_bounds_for_nf

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

inference_model = EasyInferenceModel(
    config_name=NF_CONFIG,
    device=device,
)

# Interpolate onto the model Q-grid (includes sigmas for dR-aware models)
q_model, curve_interp, sigmas_interp = inference_model.interpolate_data_to_model_q(
    q_exp, curve_exp, sigmas_exp=sigmas_exp
)

# Extend prior bounds with nuisance parameters (r_scale, log10_background)
# required by the NF model, handled automatically by _extend_prior_bounds_for_nf
prior_bounds_nf = _extend_prior_bounds_for_nf(prior_bounds)

prediction_dict = inference_model.preprocess_and_sample(
    reflectivity_curve=curve_interp,
    q_values=q_model,
    num_samples=NF_SAMPLES,
    prior_bounds=prior_bounds_nf,
    q_resolution=0.1,
    calc_sampled_curves=True,
    calc_sampled_sld_profiles=True,
    calc_log_likelihoods=True,
    enable_importance_sampling=True,
)

print("Inference complete.")
print(f"Sampled params shape : {prediction_dict['predicted_params_array'].shape}")
print(f"Parameter names      : {prediction_dict['param_names']}")


Configuration file `/home/levytskyi/Documents/TEMP_thesis_code_testing/evaluation_pipeline/vendor/nflows_reflectorch/configs/example_nf_config_reflectorch.yaml` found locally.
Weights file `/home/levytskyi/Documents/TEMP_thesis_code_testing/evaluation_pipeline/vendor/nflows_reflectorch/saved_models/model_example_nf_config_reflectorch.safetensors` found locally.
Model example_nf_config_reflectorch.yaml loaded. Number of parameters: 40.91 M
The model corresponds to a `standard_model` parameterization with 1 layers (7 predicted parameters)
Parameter types and total ranges:
- thicknesses: [1.0, 1500.0]
- roughnesses: [0.0, 60.0]
- slds: [-8.0, 16.0]
- r_scale: [0.9, 1.1]
- log10_background: [-10.0, -4.0]
Allowed widths of the prior bound intervals (max-min):
- thicknesses: [0.01, 1500.0]
- roughnesses: [0.01, 60.0]
- slds: [0.01, 24.0]
- r_scale: [0.001, 0.2]
- log10_background: [0.01, 6.0]
The model was trained on curves discretized at exactly 256 uniform points, between q_min in [0.001, 

ValueError: 
 **Parameter dimension mismatch during inference!**
- Model expects **7** parameters.
- You provided **5** prior bounds.

💡This often occurs when:
- The model was trained with additional nuisance parameters like `r_scale`, `q_shift`, or `log10_background`,
  but they were not included in the `prior_bounds` passed to `.predict()`.
- The number of layers or parameterization type differs from the one used during training.

 Check the configuration or the summary of expected parameters.

## 7. Extract MAP Estimate

Select the highest log-likelihood sample as the MAP (maximum a posteriori) estimate.

In [ ]:
from nf_statistics import compute_nf_sample_statistics

log_likelihoods = prediction_dict["log_likelihoods"]
pred_params     = prediction_dict["predicted_params_array"]
nf_param_names  = prediction_dict["param_names"]

best_idx  = int(np.argmax(log_likelihoods))
map_params = pred_params[best_idx]

print(f"MAP sample index     : {best_idx}")
print(f"MAP log-likelihood   : {log_likelihoods[best_idx]:.4f}")
print()

stats = compute_nf_sample_statistics(pred_params, nf_param_names)

print(f"{'Parameter':<20}  {'True':>10}  {'MAP':>10}  {'Mean':>10}  {'Std':>10}")
print("-" * 66)
for i, name in enumerate(nf_param_names):
    print(
        f"{name:<20}  {true_params[i]:>10.4f}  {map_params[i]:>10.4f}"
        f"  {stats['mean'][i]:>10.4f}  {stats['std'][i]:>10.4f}"
    )

## 8. Visualise Reflectivity Curve Fit and SLD Profile

`plot_simple_comparison` produces a two-panel figure: the experimental curve with the MAP fit overlaid, and the corresponding SLD depth profile.

In [ ]:
from plotting_utils import plot_simple_comparison

# Build a minimal results dict matching simple_pipeline output format
results = {
    "prediction_dict": prediction_dict,
    "q_exp": q_exp,
    "curve_exp": curve_exp,
    "sigmas_exp": sigmas_exp,
    "true_params": true_params,
    "param_names": nf_param_names,
    "prior_bounds": prior_bounds,
    "layer_count": LAYER_COUNT,
    "experiment_id": EXPERIMENT_ID,
}

plot_simple_comparison(results, save=False)

## 9. Compute Constraint MAPE Metrics

Constraint MAPE normalises the absolute prediction error by the prior half-width rather than the true value:

$$\text{Constraint MAPE} = \frac{|\hat{y} - y|}{\text{constraint width}} \times 100$$

A value of 0 means the prediction equals the true value; 100 means the prediction is at the constraint boundary.

In [ ]:
from error_calculation import calculate_parameter_metrics, print_metrics_report

param_metrics = calculate_parameter_metrics(
    predicted_params=map_params,
    true_params=true_params,
    param_names=nf_param_names,
    prior_bounds=prior_bounds,
    priors_type="constraint_based",
)

print_metrics_report(param_metrics)

## 10. One-Line Pipeline Shortcut

The above steps are equivalent to calling `run_single_experiment` directly, which is the recommended entry point for scripted use.

In [ ]:
from simple_pipeline import run_single_experiment

# All of the above condensed into one call
full_results = run_single_experiment(
    experiment_id=EXPERIMENT_ID,
    layer_count=LAYER_COUNT,
    enable_preprocessing=ENABLE_PREPROCESSING,
    priors_deviation=PRIORS_DEV / 100.0,
    nf_config_name=NF_CONFIG,
    nf_num_samples=NF_SAMPLES,
)

overall_mape = full_results["param_metrics"]["overall"]["constraint_mape"]
print(f"Overall Constraint MAPE: {overall_mape:.2f}%")